# Search Path and Fallback Story

This notebook sits between the existing ingestion and Landsat exploration notebooks.

It tells a simple story:
1. We start with the normal Phase A path: search for Sentinel-2 scenes for one AOI.
2. We verify that the shared STAC search code can reach a working endpoint.
3. We inspect the first returned item to confirm the assets we expect are really there.
4. If Sentinel-2 returns nothing, we show the fallback path to Landsat.

This is not the notebook that reads rasters or computes indices. That work already lives in the Sentinel ingestion notebook. This notebook is only about proving that the discovery step is healthy and understandable.

## How this notebook fits with the others

Use the notebooks as a sequence:
- `00-sentinel2_ingestion_timeseries.ipynb`: full Sentinel-2 ingestion, AOI window reads, masking, indices, and saved outputs.
- `01-search-path-and-fallback-story.ipynb`: readable verification of the STAC search path and fallback behavior.
- `00-landsat8_exploration.ipynb`: deeper Landsat-specific exploration once fallback data is actually needed.
- `ndvi_over_time.ipynb`: quick inspection of saved time-series outputs.

That means this notebook expands the existing set without replacing any of them.

## 1) Configuration

All tunables stay in one place so the rest of the notebook reads like a walkthrough instead of a hunt for hidden constants.

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

# Time window for the search story: by default we look back 24 months from now.
NOW_UTC: datetime = datetime.now(timezone.utc)
LOOKBACK_MONTHS: int = 24
END_DATE_UTC: datetime = NOW_UTC

# AOI fixtures: keep both polygon and point modes so the notebook can explain both paths.
AOI_MODE: str = "polygon"
AOI_POLYGON_GEOJSON: Dict[str, Any] = {
    "type": "Polygon",
    "coordinates": [[
        [31.0000, 30.0000],
        [31.0100, 30.0000],
        [31.0100, 30.0100],
        [31.0000, 30.0100],
        [31.0000, 30.0000],
    ]],
}
AOI_POINT_LON_LAT: Tuple[float, float] = (31.0000, 30.0000)
POINT_BUFFER_RADIUS_M: int = 500

# Endpoint order matters in auto mode: we try these in sequence until one works.
STAC_ENDPOINT_MODE: str = "auto"
STAC_ENDPOINTS: List[Dict[str, Any]] = [
    {
        "name": "planetary_computer",
        "api_url": "https://planetarycomputer.microsoft.com/api/stac/v1",
        "supports_signing": True,
    },
    {
        "name": "earth_search",
        "api_url": "https://earth-search.aws.element84.com/v1",
        "supports_signing": False,
    },
]

# Sentinel-2 is the primary source; Landsat is only the fallback path.
SENTINEL2_COLLECTION: str = "sentinel-2-l2a"
LANDSAT_COLLECTION: str = "landsat-c2-l2"

# Search controls keep the notebook easy to tune without editing later cells.
MAX_ITEMS: int = 50
USE_CLOUD_FILTER: bool = True
MAX_CLOUD_COVER: float = 40.0
REQUIRED_ASSET_KEYS: List[str] = ["B02", "B03", "B04", "B08"]

## 2) Imports and shared helpers

The important design choice here is that the notebook does not reimplement the STAC client logic. It imports the shared search helpers from `farmtrust_core` so the evidence matches the real code path.

In [ ]:
import importlib

# Collect missing packages so the notebook can fail with a useful message.
missing_pip: List[str] = []

def _try_import(module: str, pip_name: Optional[str] = None):
    try:
        return importlib.import_module(module)
    except Exception:
        missing_pip.append(pip_name or module)
        return None

# Only import what this notebook actually needs for search-path verification.
pystac_client = _try_import("pystac_client", "pystac-client")
shapely = _try_import("shapely", "shapely")
pyproj = _try_import("pyproj", "pyproj")

# STAC search is the one hard requirement for this notebook.
if pystac_client is None:
    raise RuntimeError("pystac-client is required. Install it with: uv add pystac-client")

# Reuse the shared production-facing helpers instead of reimplementing search logic here.
from farmtrust_core.ingest.stac_client import compute_start_date, stac_search_items

## 3) Build the search window and AOI geometry

Before we search, we need two things:
- a time window in RFC3339 format
- an `intersects` geometry that STAC understands

If the AOI is a point, we try to turn it into a small polygon buffer so the search area is more realistic.

In [ ]:
from typing import Callable

# Convert datetimes into the RFC3339 strings expected by STAC APIs.
def _to_rfc3339(dt: datetime) -> str:
    dt = dt.astimezone(timezone.utc).replace(microsecond=0)
    return dt.isoformat().replace("+00:00", "Z")

def datetime_range_rfc3339(start_dt: datetime, end_dt: datetime) -> str:
    return f"{_to_rfc3339(start_dt)}/{_to_rfc3339(end_dt)}"

# Build the AOI geometry from the selected notebook fixture.
def get_aoi_geometry() -> Dict[str, Any]:
    if AOI_MODE == "polygon":
        return AOI_POLYGON_GEOJSON
    if AOI_MODE == "point":
        lon, lat = AOI_POINT_LON_LAT
        return {"type": "Point", "coordinates": [lon, lat]}
    raise ValueError(f"Unsupported AOI_MODE: {AOI_MODE}")

# If geometry libraries are available, turn a point into a small polygon in meters.
def buffer_point_to_polygon_if_possible(point_geojson: Dict[str, Any], radius_m: int) -> Dict[str, Any]:
    if shapely is None or pyproj is None:
        return point_geojson

    from shapely.geometry import Point, mapping
    from shapely.ops import transform
    from pyproj import CRS, Transformer

    lon, lat = point_geojson["coordinates"]
    wgs84 = CRS.from_epsg(4326)
    aeqd = CRS.from_proj4(
        f"+proj=aeqd +lat_0={lat} +lon_0={lon} +x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
    )

    to_aeqd: Callable[[float, float], Tuple[float, float]] = Transformer.from_crs(
        wgs84, aeqd, always_xy=True
    ).transform
    to_wgs84: Callable[[float, float], Tuple[float, float]] = Transformer.from_crs(
        aeqd, wgs84, always_xy=True
    ).transform

    p = Point(lon, lat)
    p_proj = transform(to_aeqd, p)
    poly_proj = p_proj.buffer(radius_m)
    poly_wgs84 = transform(to_wgs84, poly_proj)
    return mapping(poly_wgs84)

# This is the geometry that will actually be sent to the STAC search.
def build_intersects_geometry() -> Dict[str, Any]:
    geom = get_aoi_geometry()
    if geom["type"] == "Point" and POINT_BUFFER_RADIUS_M > 0:
        return buffer_point_to_polygon_if_possible(geom, POINT_BUFFER_RADIUS_M)
    return geom

# Keep endpoint selection notebook-local, but simple and explicit.
def choose_endpoints(mode: str) -> List[Dict[str, Any]]:
    if mode == "auto":
        return STAC_ENDPOINTS
    for endpoint in STAC_ENDPOINTS:
        if endpoint["name"] == mode:
            return [endpoint]
    raise ValueError(f"Unknown STAC_ENDPOINT_MODE={mode}")

# Build the optional cloud-cover filter only when the config enables it.
def build_cloud_query() -> Optional[Dict[str, Any]]:
    if not USE_CLOUD_FILTER:
        return None
    return {"eo:cloud_cover": {"lt": MAX_CLOUD_COVER}}

# Materialize the effective search inputs once so later cells can focus on the story.
START_DATE_UTC = compute_start_date(END_DATE_UTC, LOOKBACK_MONTHS)
DATETIME_RFC3339 = datetime_range_rfc3339(START_DATE_UTC, END_DATE_UTC)
INTERSECTS = build_intersects_geometry()
ENDPOINTS = choose_endpoints(STAC_ENDPOINT_MODE)
QUERY = build_cloud_query()

print(DATETIME_RFC3339)
INTERSECTS

## 4) First chapter: can we find Sentinel-2 scenes?

This is the normal path. If this succeeds, the ingestion notebook has something real to work with later. If it fails, we want the failure to be readable: which endpoint was used, how many items came back, and whether the first item looks structurally correct.

In [ ]:
# Reduce a list of STAC items into a compact evidence block for the notebook narrative.
def summarize_items(items: List[Any], label: str) -> Dict[str, Any]:
    summary: Dict[str, Any] = {
        "label": label,
        "count": len(items),
        "first_item_id": None,
        "first_item_datetime": None,
        "asset_keys_preview": [],
        "missing_required_assets": [],
    }

    if not items:
        return summary

    # We only inspect the first item because this notebook is about search-path health, not full analysis.
    first_item = items[0]
    asset_keys = list((first_item.assets or {}).keys())
    missing_assets = [key for key in REQUIRED_ASSET_KEYS if key not in (first_item.assets or {})]

    summary["first_item_id"] = first_item.id
    summary["first_item_datetime"] = (first_item.properties or {}).get("datetime")
    summary["asset_keys_preview"] = asset_keys[:12]
    summary["missing_required_assets"] = missing_assets
    return summary

# Run the primary Sentinel-2 search through the shared helper path.
sentinel_items, sentinel_endpoint = stac_search_items(
    endpoints=ENDPOINTS,
    collection=SENTINEL2_COLLECTION,
    intersects_geojson=INTERSECTS,
    datetime_range=DATETIME_RFC3339,
    limit=MAX_ITEMS,
    query=QUERY,
)

# Store the result in a compact structure that later cells can print or compare.
SENTINEL_SUMMARY = summarize_items(sentinel_items, "Sentinel-2 primary")
SENTINEL_SUMMARY["endpoint"] = sentinel_endpoint
SENTINEL_SUMMARY

## 5) Reading the result

At this point we are not asking whether the imagery is scientifically useful yet. We are asking a simpler operational question: did the shared search path return a plausible scene from a plausible endpoint, with the assets the ingestion pipeline expects?

If the answer is yes, the heavier ingestion notebook can proceed with confidence.

In [ ]:
if SENTINEL_SUMMARY["count"] > 0:
    print("Sentinel-2 search succeeded.")
    print(f"Endpoint: {SENTINEL_SUMMARY['endpoint']}")
    print(f"Items found: {SENTINEL_SUMMARY['count']}")
    print(f"First item: {SENTINEL_SUMMARY['first_item_id']}")
    print(f"Datetime: {SENTINEL_SUMMARY['first_item_datetime']}")
    print(f"Asset preview: {SENTINEL_SUMMARY['asset_keys_preview']}")
    if SENTINEL_SUMMARY["missing_required_assets"]:
        print(f"Missing expected assets: {SENTINEL_SUMMARY['missing_required_assets']}")
    else:
        print("All expected Sentinel-2 assets are present.")
else:
    print("Sentinel-2 returned no items for this AOI and time window.")

## 6) Second chapter: what happens if Sentinel-2 is empty?

Phase A treats Landsat as a fallback, not a co-equal primary source. That means we only go here when Sentinel-2 gives us nothing useful for the configured search window.

This keeps the story honest: Sentinel-2 first, Landsat only when needed.

In [ ]:
# Default to no fallback result; we only populate this if Sentinel-2 is empty.
LANDSAT_SUMMARY: Optional[Dict[str, Any]] = None

# Landsat is a true fallback in this story, so we only search it when the primary path fails.
if SENTINEL_SUMMARY["count"] == 0:
    landsat_items, landsat_endpoint = stac_search_items(
        endpoints=ENDPOINTS,
        collection=LANDSAT_COLLECTION,
        intersects_geojson=INTERSECTS,
        datetime_range=DATETIME_RFC3339,
        limit=MAX_ITEMS,
        query=QUERY,
    )
    LANDSAT_SUMMARY = summarize_items(landsat_items, "Landsat fallback")
    LANDSAT_SUMMARY["endpoint"] = landsat_endpoint

LANDSAT_SUMMARY

## 7) Final evidence block

The point of this notebook is to leave behind a compact, readable proof of what happened. This summary is the handoff to the rest of the notebook set.

In [ ]:
# Final handoff object: this is the shortest durable record of what the notebook proved.
STORY_SUMMARY = {
    "datetime_window": DATETIME_RFC3339,
    "endpoint_mode": STAC_ENDPOINT_MODE,
    "sentinel": SENTINEL_SUMMARY,
    "landsat_fallback": LANDSAT_SUMMARY,
    "next_notebook": (
        "00-sentinel2_ingestion_timeseries.ipynb"
        if SENTINEL_SUMMARY["count"] > 0
        else "00-landsat8_exploration.ipynb"
    ),
}

STORY_SUMMARY

## What this notebook adds

This notebook does not replace the existing ones. It adds a readable bridge between them:
- it explains the search step before raster work begins
- it proves the shared STAC path is working
- it shows the fallback decision clearly
- it points the reader to the next notebook depending on the outcome

That makes the notebook set easier to read as a story instead of a pile of disconnected experiments.